# NovaBank Digital Supervised Baseline And Account-Takeover Threshold Tuning

Fit a small supervised baseline on NovaBank Digital feature-library outputs, then
tune alert thresholds framed around **account-takeover (ATO)** detection.

The notebook consumes `build_digital_banking_features()` output and the shared
alert-aware evaluation utility. It avoids headline accuracy claims, keeps
protected scenario answer keys separate from the learner-facing investigation
path, and interprets false positives as legitimate-looking activity that an ATO
review would still need to investigate.

**Glossary reminder:** a **Client** is the legal customer, a **User** is the
digital login identity, and account-takeover behavior is modeled under the
`new_beneficiary_payment` and `session_payment_velocity` Detection patterns.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from banking_fraud_lab import (
    build_digital_banking_features,
    build_learner_facing_views,
    evaluate_alert_scores,
    generate_digital_fraud_scenarios_world,
)
from banking_fraud_lab.features import DIGITAL_BANKING_FEATURE_FAMILIES
from banking_fraud_lab.generators.digital_banking import (
    ACCOUNT_TAKEOVER_BENEFICIARY_ACTIVITY_TYPE,
    ACCOUNT_TAKEOVER_VELOCITY_ACTIVITY_TYPE,
)
from banking_fraud_lab.schema import PATTERN_IDS, PROTECTED_SCENARIO_ANSWER_KEYS

pd.set_option("display.max_columns", 40)

## Generate Learner-Facing Data

The supervised label comes from generated case outcomes. Protected scenario
answer keys stay outside the learner-facing views, so the investigation path
never sees a clean ground-truth label. Noisy outcomes (where confirmed-fraud
disagrees with the true protected label) are part of the data on purpose.

In [ ]:
tables = generate_digital_fraud_scenarios_world(
    seed=42,
    scale="small",
    scenario_prevalence=0.5,
    noisy_outcome_rate=0.3,
)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables

learner_summary = pd.DataFrame(
    {
        "table": ["cases", "case_outcomes", "alerts", "transactions"],
        "rows": [
            len(learner_tables["cases"]),
            len(learner_tables["case_outcomes"]),
            len(learner_tables["alerts"]),
            len(learner_tables["transactions"]),
        ],
    }
)
learner_summary

## Load Feature-Library Outputs

The model consumes `build_digital_banking_features()` output. Feature-family
metadata keeps inputs traceable to Detection pattern identifiers, and the v0.4
digital families map only to existing digital pattern IDs.

In [ ]:
feature_frame = build_digital_banking_features(learner_tables)
feature_family_map = pd.DataFrame(
    [
        {
            "family_id": spec.family_id,
            "display_name": spec.display_name,
            "detection_pattern_id": spec.detection_pattern_id,
        }
        for spec in DIGITAL_BANKING_FEATURE_FAMILIES
    ]
)

assert set(feature_family_map["detection_pattern_id"]).issubset(set(PATTERN_IDS))
assert set(feature_family_map["detection_pattern_id"]) <= {
    "digital_scam_to_mule",
    "new_beneficiary_payment",
    "session_payment_velocity",
}
feature_family_map

## Build The Supervised Modeling Frame

Case outcomes provide labels after the Alert lifecycle has completed. The frame
joins cases to feature-library outputs on the transaction under investigation.

In [ ]:
model_frame = (
    learner_tables["cases"][["case_id", "alert_id", "transaction_id"]]
    .merge(
        learner_tables["alerts"][["alert_id", "alert_type", "severity", "reason"]],
        on="alert_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        learner_tables["case_outcomes"][["case_id", "confirmed_fraud", "loss_amount_chf"]],
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(feature_frame, on="transaction_id", how="inner", validate="one_to_one")
)

assert model_frame["alert_id"].notna().all()
assert model_frame["confirmed_fraud"].notna().all()
assert model_frame["alert_id"].is_unique
assert model_frame["confirmed_fraud"].nunique() == 2

ato_types = {
    ACCOUNT_TAKEOVER_BENEFICIARY_ACTIVITY_TYPE,
    ACCOUNT_TAKEOVER_VELOCITY_ACTIVITY_TYPE,
}
ato_share = model_frame["alert_type"].isin(ato_types).mean()
model_frame[
    [
        "case_id",
        "alert_id",
        "alert_type",
        "confirmed_fraud",
        "db_is_new_beneficiary",
        "db_is_shared_device",
        "db_is_rapid_pass_through",
        "db_session_payment_count",
        "score" if "score" in model_frame.columns else "amount_chf",
    ]
].head(10)

## Fit A Reproducible Baseline

The pipeline standardizes numeric feature-library outputs and fits a
logistic-regression baseline. The fit uses tiny data on purpose so the notebook
stays reproducible; the goal is the threshold-tuning workflow, not a production
model.

In [ ]:
feature_output_columns = [
    output_column
    for spec in DIGITAL_BANKING_FEATURE_FAMILIES
    for output_column in spec.output_columns
]
numeric_feature_columns = [
    column
    for column in feature_output_columns
    if pd.api.types.is_numeric_dtype(model_frame[column])
]
assert numeric_feature_columns

preprocess = ColumnTransformer(
    [("numeric", StandardScaler(), numeric_feature_columns)],
    remainder="drop",
)
baseline_model = Pipeline(
    [
        ("preprocess", preprocess),
        (
            "model",
            LogisticRegression(
                random_state=42,
                solver="lbfgs",
                max_iter=1000,
                class_weight="balanced",
            ),
        ),
    ]
)
target = model_frame["confirmed_fraud"].astype(bool)
baseline_model.fit(model_frame[numeric_feature_columns], target)

model_frame = model_frame.assign(
    score=baseline_model.predict_proba(model_frame[numeric_feature_columns])[:, 1].round(6)
)
alert_scores = model_frame[["alert_id", "score"]]
alert_scores

## Connect Coefficients To Detection Patterns

Coefficient direction is a compact way to inspect the tiny baseline. The
feature-family mapping keeps each coefficient traceable to a Detection pattern,
so ATO-relevant signals (new beneficiary, session velocity, shared device) are
visible alongside the scam-to-mule signals.

In [ ]:
coefficients = pd.DataFrame(
    {
        "feature": numeric_feature_columns,
        "coefficient": baseline_model.named_steps["model"].coef_[0],
    }
)
coefficients = coefficients.sort_values("coefficient", ascending=False).reset_index(
    drop=True
)
coefficients

## Tune Thresholds With Capacity And Costs

Thresholds are evaluated through the shared alert-aware utility. This keeps
PR-AUC, precision, recall, alert volume, capacity utilization, and total cost on
one learner-facing surface. The investigation capacity, false-positive cost, and
missed-fraud cost parameters model the operational constraints of an ATO review
queue: a low threshold floods analysts with alerts, while a high threshold lets
real takeover payments through.

In [ ]:
report = evaluate_alert_scores(
    cases=model_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=alert_scores,
    thresholds=(0.85, 0.75, 0.60, 0.40, 0.25),
    alert_capacity=6,
    investigation_cost_chf=200.0,
    false_positive_cost_chf=600.0,
    missed_fraud_cost_chf=40_000.0,
)
required_report_fields = {
    "pr_auc",
    "threshold_summaries",
    "cost_curve",
    "lowest_cost_threshold",
    "limitation_summary",
}
assert required_report_fields <= set(report)

threshold_summaries = pd.DataFrame(report["threshold_summaries"])
threshold_summaries[
    [
        "threshold",
        "precision",
        "recall",
        "alert_volume",
        "alert_capacity",
        "capacity_utilization",
        "over_capacity_alerts",
        "false_positives",
        "false_negatives",
        "total_cost_chf",
    ]
]

## Compare Cost Curve And Lowest-Cost Threshold

A high threshold can miss confirmed ATO fraud, while a low threshold can create
false positives and exceed investigation capacity. The lowest-cost threshold
balances these against the cost curve.

In [ ]:
cost_curve = pd.DataFrame(report["cost_curve"])
overview = pd.DataFrame(
    [
        {
            "pr_auc": report["pr_auc"],
            "lowest_cost_threshold": report["lowest_cost_threshold"],
            "case_count": report["population"]["case_count"],
            "confirmed_fraud_count": report["population"]["confirmed_fraud_count"],
        }
    ]
)

assert threshold_summaries["alert_capacity"].notna().all()
assert threshold_summaries["capacity_utilization"].notna().all()

overview.merge(
    cost_curve,
    left_on="lowest_cost_threshold",
    right_on="threshold",
    how="left",
    validate="one_to_one",
)

## Inspect Account-Takeover False Positives

Account-takeover review surfaces legitimate-looking activity that the baseline
scores highly even though the case outcome did not confirm fraud. These are the
false positives an ATO investigation queue would still need to clear, which is
why false-positive cost and capacity matter for threshold tuning. The protected
answer keys are not used here — this path works only from learner-facing case
outcomes.

In [ ]:
ato_false_positive_examples = model_frame[
    model_frame["alert_type"].isin(ato_types)
    & ~model_frame["confirmed_fraud"].astype(bool)
].copy()
assert not ato_false_positive_examples.empty

ato_false_positive_examples[
    [
        "case_id",
        "alert_id",
        "alert_type",
        "db_is_new_beneficiary",
        "db_is_shared_device",
        "db_session_payment_count",
        "db_account_age_days",
        "score",
    ]
]

## Keep Evaluation Limits Visible

The report's limitation wording is part of the learner-facing output. It keeps
the tiny supervised baseline focused on alert decisions and operational
tradeoffs instead of headline accuracy. Confirmed-fraud is an imperfect, noisy
label in this track, so accuracy claims would be misleading.

In [ ]:
assert "accuracy is out of scope" in report["limitation_summary"]

pd.DataFrame(
    [
        {
            "pr_auc": report["pr_auc"],
            "lowest_cost_threshold": report["lowest_cost_threshold"],
            "limitation_summary": report["limitation_summary"],
        }
    ]
)